# Previsão diária de acidentes rodoviários por tipo em Portugal

**Objetivo.** Prever separadamente o número diário de despistes, colisões e atropelamentos registados no ficheiro nacional, incluindo os acidentes das ilhas presentes na fonte.

1. Com que erro consegue um modelo de machine learning prever a contagem diária de cada tipo em anos fora do treino?
2. Para que tipos melhora face a uma referência histórica baseada no mês e no dia da semana?
3. Em que medida a meteorologia e a luz natural acrescentam capacidade preditiva?

Este notebook prepara os dados, descreve as distribuições e faz diagnósticos com regressões de Poisson. As comparações exploratórias em validação orientam a modelação posterior. O teste de 2024 fica reservado para a comparação final, incluindo a futura rede neuronal.

**Unidade e horizonte.** Uma linha por data civil da fonte, a nível nacional. O dia é o índice e origina variáveis de calendário. As três respostas são as contagens do dia-alvo. O histórico usa datas anteriores e a disponibilidade atempada dos registos tem de ser confirmada. A meteorologia ERA5 do próprio dia é reanálise e permite uma avaliação retrospetiva. Para uma previsão operacional, esse bloco requer previsões meteorológicas arquivadas que estivessem disponíveis na hora de emissão.

**Reprodução.** Abrir com um kernel Python com pandas, numpy, scipy, matplotlib, statsmodels e ipykernel. O notebook localiza a raiz do projeto a partir da pasta de execução. A aquisição e as fontes estão documentadas em [Meteorologia diária](../docs/meteo_diaria.md).

In [ ]:
from pathlib import Path
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy.special import xlogy
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "aquisicao").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Executar o notebook dentro da pasta do projeto.")

ACIDENTES = ROOT / "data_agregada" / "acidentes.csv"
METEO = ROOT / "meteo" / "meteorologia_diaria_portugal.csv"
if not METEO.is_file():
    raise FileNotFoundError("Executar python aquisicao/obter_meteo_diaria.py antes da EDA.")

ANOS_TREINO = list(range(2010, 2020))
ANOS_FONTE = ANOS_TREINO + [2023, 2024]
TIPOS = ["Despiste", "Colisão", "Atropelamento"]
DIAS_SEMANA = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
CORES = dict(zip(TIPOS, ["#00747A", "#3459A6", "#B44429"]))
plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False, "axes.spines.right": False})

## 1. Construção e auditoria das contagens

As modalidades detalhadas observadas são mapeadas para as três famílias usando a classificação de `Natureza` do [dicionário de acidentes](../docs/dicionarios/acidentes.md). Atropelamentos de animais e com fuga entram na família de atropelamentos. A unidade a contar é a chave `(ano, Id. Acidente)`. Nos anos com ficheiro, um dia sem registos recebe zero. A cobertura da fonte continua a limitar a interpretação dessas contagens. O calendário completo preserva 2020–2022 como ausentes e impede que os atrasos façam a ponte de 2019 para 2023.

In [ ]:
acidentes = pd.read_csv(
    ACIDENTES,
    usecols=["ano", "Id. Acidente", "Datahora", "Distrito", "Natureza"],
    dtype={"Id. Acidente": "string"},
)
assert set(acidentes["ano"]) == set(ANOS_FONTE), "Rever cobertura de anos da fonte."
assert acidentes["Id. Acidente"].notna().all() and acidentes["Id. Acidente"].str.strip().ne("").all()
assert not acidentes.duplicated(["ano", "Id. Acidente"]).any(), "Investigar chaves repetidas."

datahora = pd.to_datetime(
    acidentes["Datahora"].str.strip(), format="%Y:%m:%d %H:%M:%S", errors="coerce"
)
assert datahora.notna().all() and datahora.dt.year.eq(acidentes["ano"]).all()
acidentes["data"] = datahora.dt.normalize()

modalidades = {
    "Despiste": [
        "Despiste simples", "Despiste com capotamento", "Despiste sem dispositivo de retenção",
        "Despiste com colisão com veículo imobil. ou obstáculo",
        "Despiste com dispositivo de retenção", "Despiste com fuga",
        "Despiste com transposição do dispositivo de retenção lateral",
    ],
    "Colisão": [
        "Colisão lateral com outro veículo em movimento",
        "Colisão traseira com outro veículo em movimento", "Colisão frontal",
        "Colisão com outras situações", "Colisão com veiculo ou obstáculo na faixa de rodagem",
        "Colisão choque em cadeia", "Colisão com fuga",
    ],
    "Atropelamento": [
        "Atropelamento de peões", "Atropelamento com fuga", "Atropelamento de animais",
    ],
}
mapa = {modalidade: tipo for tipo, nomes in modalidades.items() for modalidade in nomes}
acidentes["natureza"] = acidentes["Natureza"].str.strip()
acidentes["tipo"] = acidentes["natureza"].map(mapa)
assert acidentes["tipo"].notna().all(), acidentes.loc[acidentes["tipo"].isna(), "natureza"].unique()

calendario = pd.date_range("2010-01-01", "2024-12-31", freq="D", name="data")
observado = calendario.year.isin(ANOS_FONTE)
contagens = pd.crosstab(acidentes["data"], acidentes["tipo"])
contagens = contagens.reindex(index=calendario, columns=TIPOS)
contagens.loc[observado] = contagens.loc[observado].fillna(0)
contagens = contagens.astype("Int64")
assert contagens.index.is_unique and contagens.notna().all(axis=1).eq(observado).all()
assert contagens.ge(0).fillna(True).to_numpy().all()
assert contagens.sum().sum() == len(acidentes)
assert contagens.sum().reindex(TIPOS).astype(int).tolist() == (
    acidentes["tipo"].value_counts().reindex(TIPOS).tolist()
)

acidentes_treino = acidentes[acidentes["ano"].isin(ANOS_TREINO)]
display(acidentes_treino.groupby(["tipo", "natureza"]).size().rename("Acidentes no treino").to_frame())
display(pd.DataFrame({
    "Período": ["2010–2019", "2020–2022", "2023", "2024"],
    "Estado": ["Desenvolvimento", "Sem ficheiro de acidentes", "Validação temporal", "Teste reservado"],
    "Dias": [calendario.year.isin(ANOS_TREINO).sum(), (~observado).sum(), (calendario.year == 2023).sum(),
             (calendario.year == 2024).sum()],
}))
print(f"{len(acidentes):,} acidentes reconciliados em {observado.sum():,} dias observados.")

## 2. Meteorologia e luz à escala nacional

A fonte ERA5 fornece um ponto por distrito do continente e por ilha, referenciado pelo IPMA. A média nacional de cada indicador usa pesos fixos com base na proporção de acidentes registados por área em **2010–2014**. O objetivo é dar maior representação ao clima das áreas onde historicamente se concentram os acidentes. Estes pesos usam um período anterior a todas as validações e ficam fixos em todo o estudo. Representam a distribuição histórica dos registos, sem medir o volume de tráfego ou a exposição em cada área.

A precipitação nacional é uma média ponderada dos totais diários locais, em mm. A extensão da chuva é a percentagem do peso histórico em áreas com pelo menos 1 mm nesse dia. Esse limiar exploratório distingue precipitação ligeira de um dia chuvoso no ponto, e a percentagem é ponderada pelos registos históricos. O vento nacional é a média ponderada dos máximos locais. Mostramos também a diferença média face a pesos iguais entre as áreas representadas.

In [ ]:
def chave_area(valor):
    normalizado = unicodedata.normalize("NFKD", valor.strip().casefold())
    return "".join(c for c in normalizado if not unicodedata.combining(c))


meteo = pd.read_csv(METEO, parse_dates=["data"])
meteo["area_id"] = meteo["area"].map(chave_area)
acidentes["area_id"] = acidentes["Distrito"].map(chave_area)
assert meteo["data"].notna().all() and not meteo.duplicated(["area_id", "data"]).any()
assert set(meteo["data"]) == set(calendario)
assert meteo.groupby("tipo_area")["area_id"].nunique().to_dict() == {"distrito": 18, "ilha": 11}
assert meteo.groupby("area_id")["data"].size().eq(len(calendario)).all()
assert set(acidentes["area_id"]) <= set(meteo["area_id"])

meteo["duracao_dia_h"] = meteo["duracao_dia_s"] / 3600
meteo["sol_direto_h"] = meteo["horas_sol_s"] / 3600
limites = {
    "temperatura_media_c": (-50, 60), "precipitacao_total_mm": (0, np.inf),
    "vento_max_kmh": (0, np.inf), "nebulosidade_media_pct": (0, 100),
    "duracao_dia_h": (0, 24), "sol_direto_h": (0, 24),
    "radiacao_total_mj_m2": (0, np.inf),
}
qualidade = pd.DataFrame({
    "Omissos": meteo[list(limites)].isna().sum(),
    "Fora de limites": {
        c: (~meteo[c].between(*intervalo) & meteo[c].notna()).sum()
        for c, intervalo in limites.items()
    },
})
display(qualidade)
assert qualidade.to_numpy().sum() == 0, "Investigar cobertura e valores antes de agregar."

pesos = acidentes.loc[acidentes["ano"].between(2010, 2014), "area_id"].value_counts(normalize=True)
assert np.isclose(pesos.sum(), 1)
meteo["peso"] = meteo["area_id"].map(pesos).fillna(0)
assert np.allclose(meteo.groupby("data")["peso"].sum(), 1)

meteo["areas_com_chuva_pct"] = 100.0 * meteo["precipitacao_total_mm"].ge(1.0)
meteorologicas = [*limites, "areas_com_chuva_pct"]
meteo_nacional = meteo[meteorologicas].mul(meteo["peso"], axis=0).groupby(meteo["data"]).sum()
meteo_pesos_iguais = meteo.loc[meteo["peso"].gt(0)].groupby("data")[meteorologicas].mean()
diferenca = (meteo_nacional - meteo_pesos_iguais).abs().loc["2010":"2019"].mean()
display(diferenca.round(3).rename("Diferença absoluta média vs pesos iguais (treino)").to_frame())
display(
    meteo[["area", "peso"]].drop_duplicates().sort_values("peso", ascending=False)
    .assign(peso_pct=lambda d: (100 * d["peso"]).round(2)).drop(columns="peso").reset_index(drop=True)
)
print(f"{len(meteo):,} registos de meteorologia diária em {meteo['area_id'].nunique()} áreas.")

## 3. Distribuições e escolha inicial das entradas

As tabelas e os gráficos desta secção usam 2010–2019. A assimetria e os dias sem ocorrências motivam modelos de contagem. A dispersão bruta mistura estações e dias da semana, pelo que a verificação de Poisson usa também a dispersão **após o ajuste**.

| Entrada candidata | Justificação e decisão a testar |
|---|---|
| Dia da semana | Captura diferenças semanais na circulação. Usar como categoria com segunda-feira como referência no GLM. |
| Época do ano | Seno e cosseno representam sazonalidade anual contínua. A referência histórica usa o mês. |
| Tendência temporal | Testar se a evolução de longo prazo melhora a validação. Avaliar com atenção em 2023, após a lacuna. |
| Contagem D−2 e média D−8 a D−2 | Captam a atividade recente de cada tipo. Comparar isoladamente e em conjunto. Confirmar a disponibilidade na origem da previsão. |
| Precipitação total | A chuva pode alterar as condições de circulação. Descrever os zeros e usar log(1 + mm) para reduzir a influência dos valores extremos. |
| Extensão geográfica da chuva | Uma média nacional baixa pode resultar de chuva intensa localizada. Testar a extensão como alternativa e como complemento ao total. |
| Temperatura média e vento máximo | Podem captar condições meteorológicas distintas. Testar ganho por variável e em bloco. |
| Nebulosidade, duração de sol direto e radiação | Descrevem fenómenos relacionados. Comparar como alternativas e inspecionar a redundância antes de escolher. |
| Duração astronómica do dia | Conhecida à partida. Testar se traz informação adicional à sazonalidade anual. |

`Natureza` é usada para obter as respostas. Variáveis do acidente do próprio dia, como os veículos, lesões e luminosidade registada, só são conhecidas depois da ocorrência. O calendário dos jogos do SC Braga tem um âmbito local e fica fora desta primeira análise nacional.

In [ ]:
diario = contagens.join(meteo_nacional, validate="one_to_one")
diario["dia_semana"] = diario.index.day_of_week
diario["mes"] = diario.index.month
diario["tendencia_anos"] = (diario.index - calendario[0]).days / 365.25
fase = 2 * np.pi * (diario.index.day_of_year - 1) / np.where(diario.index.is_leap_year, 366, 365)
diario["seno_anual"] = np.sin(fase)
diario["cosseno_anual"] = np.cos(fase)
diario["log_precipitacao"] = np.log1p(diario["precipitacao_total_mm"])

for tipo in TIPOS:
    anterior = diario[tipo].astype(float).shift(2)
    diario[f"{tipo}_d2"] = anterior
    diario[f"{tipo}_media7"] = anterior.rolling(7, min_periods=7).mean()

assert diario.loc["2020":"2022", TIPOS].isna().all().all()
assert diario.loc["2023-01-01":"2023-01-08", [f"{t}_media7" for t in TIPOS]].isna().all().all()
assert diario.loc["2023-01-09", [f"{t}_media7" for t in TIPOS]].notna().all()

treino = diario.loc["2010":"2019"]
alvos = treino[TIPOS].astype(float)
resumo = alvos.describe(percentiles=[.05, .25, .5, .75, .95]).T
resumo["variancia"] = alvos.var()
resumo["variancia/media"] = alvos.var() / alvos.mean()
resumo["dias_zero_pct"] = alvos.eq(0).mean() * 100
display(resumo.round(2))
display(alvos.groupby(alvos.index.year).mean().round(2).rename_axis("Ano · média por dia"))

colunas_eda = meteorologicas + [f"{t}_{h}" for t in TIPOS for h in ("d2", "media7")]
resumo_x = treino[colunas_eda].describe(percentiles=[.05, .25, .5, .75, .95]).T
resumo_x["omissos"] = treino[colunas_eda].isna().sum()
resumo_x["dias_zero_pct"] = treino[colunas_eda].eq(0).mean() * 100
display(resumo_x.round(2))

fig, ax = plt.subplots(3, 2, figsize=(12, 8), layout="constrained")
for i, tipo in enumerate(TIPOS):
    y = alvos[tipo]
    ax[i, 0].hist(y, bins=np.arange(y.max() + 2) - .5, color=CORES[tipo])
    ax[i, 0].set(title=f"{tipo} · distribuição diária", xlabel="Acidentes por dia", ylabel="Dias")
    ax[i, 1].plot(y.rolling(28).mean(), color=CORES[tipo])
    ax[i, 1].set(title=f"{tipo} · média móvel de 28 dias", ylabel="Acidentes por dia")
plt.show()

fig, ax = plt.subplots(3, 2, figsize=(11, 7), layout="constrained")
for i, tipo in enumerate(TIPOS):
    y = alvos[tipo]
    ax[i, 0].bar(DIAS_SEMANA, y.groupby(y.index.day_of_week).mean(), color=CORES[tipo])
    ax[i, 1].plot(range(1, 13), y.groupby(y.index.month).mean(), marker="o", color=CORES[tipo])
    ax[i, 1].set(xticks=range(1, 13), xlabel="Mês")
    ax[i, 0].set(ylabel=f"{tipo}\nMédia por dia")
ax[0, 0].set_title("Perfil semanal")
ax[0, 1].set_title("Perfil mensal")
plt.show()

In [ ]:
nomes = {
    "temperatura_media_c": "Temperatura (°C)", "precipitacao_total_mm": "Precipitação (mm)",
    "vento_max_kmh": "Vento (km/h)", "nebulosidade_media_pct": "Nebulosidade (%)",
    "duracao_dia_h": "Duração do dia (h)", "sol_direto_h": "Sol direto (h)",
    "radiacao_total_mj_m2": "Radiação (MJ/m²)",
    "areas_com_chuva_pct": "Extensão da chuva (% ponderada)",
}
fig, ax = plt.subplots(2, 4, figsize=(13, 5.5), layout="constrained")
for eixo, (coluna, nome) in zip(ax.flat, nomes.items()):
    eixo.hist(treino[coluna], bins=30, color="#287D80")
    eixo.set(title=nome, ylabel="Dias")
plt.show()

corr = treino[[*nomes, "seno_anual", "cosseno_anual"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9.5, 6.3), layout="constrained")
cores = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
etiquetas = [nomes.get(c, c.replace("_", " ")) for c in corr]
ax.set_xticks(range(len(corr)), etiquetas, rotation=42, ha="right")
ax.set_yticks(range(len(corr)), etiquetas)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if abs(v) > .68 else "#222")
ax.set_title("Spearman entre meteorologia, luz e sazonalidade")
fig.colorbar(cores, ax=ax, shrink=.8)
plt.show()

quartis = []
for coluna in nomes:
    classe = pd.qcut(treino[coluna], 4, duplicates="drop")
    quadro = treino.groupby(classe, observed=True)[TIPOS].mean().reset_index(drop=True)
    quadro.insert(0, "Grupo (baixo → alto)", range(1, len(quadro) + 1))
    quadro.insert(0, "Variável", nomes[coluna])
    quartis.append(quadro)
display(pd.concat(quartis, ignore_index=True).round(2))

As médias por grupos de quantis são associações brutas. Por exemplo, temperaturas mais altas coincidem com uma parte do calendário anual. A análise de redundância usa Spearman e VIF e a contribuição preditiva é avaliada com os anos em ordem temporal.

## 4. Referência histórica e regressões exploratórias

Validamos em 2017, 2018, 2019 e 2023 usando apenas anos anteriores em cada ajuste. Os modelos de cada tipo usam as mesmas datas e a mesma contagem observada. Para 2023, os primeiros oito dias ficam fora desta comparação porque falta histórico imediatamente anterior. O critério principal é MAE, em acidentes por dia, acompanhado da deviância média de Poisson. A verificação de 2023 usa as contagens anteriores desse ano à medida que o tempo avança, simulando previsões diárias sucessivas com disponibilização do histórico.

Também comparamos as entradas meteorológicas individuais com o mesmo modelo de calendário e histórico. A combinação compacta usa temperatura, precipitação, vento e radiação. Testamos o contributo da duração do dia sobre esse bloco e, separadamente, como alternativa à sazonalidade de seno e cosseno.

In [ ]:
semana = pd.get_dummies(diario["dia_semana"], prefix="semana", drop_first=True, dtype=float)
anual = ["seno_anual", "cosseno_anual"]
meteo_base = ["temperatura_media_c", "log_precipitacao", "vento_max_kmh"]
candidatas = [*meteo_base, "areas_com_chuva_pct", "nebulosidade_media_pct",
              "duracao_dia_h", "sol_direto_h", "radiacao_total_mj_m2"]
anos_validacao = [2017, 2018, 2019, 2023]


def matriz(dados, colunas):
    valores = pd.concat([semana.loc[dados.index], dados[colunas].astype(float)], axis=1)
    return sm.add_constant(valores, has_constant="add")


def ajustar(dados, tipo, colunas):
    x = matriz(dados, colunas)
    assert np.linalg.matrix_rank(x) == x.shape[1], f"Matriz sem posto completo: {colunas}"
    modelo = sm.GLM(dados[tipo].astype(float), x, family=sm.families.Poisson()).fit()
    assert modelo.converged and np.isfinite(modelo.params).all()
    return modelo


def metricas(y, previsao):
    mu = np.clip(np.asarray(previsao, dtype=float), 1e-10, None)
    y = np.asarray(y, dtype=float)
    return {
        "MAE": np.abs(y - mu).mean(),
        "Deviância média": 2 * np.mean(xlogy(y, y / mu) - y + mu),
    }


def variantes(tipo):
    historico = [f"{tipo}_d2", f"{tipo}_media7"]
    base = [*anual, "tendencia_anos", *historico]
    externo = [*meteo_base, "radiacao_total_mj_m2"]
    return {
        "Calendário": anual,
        "Calendário + histórico": [*anual, *historico],
        "Calendário + tendência": [*anual, "tendencia_anos"],
        "Base sem D−2": [*anual, "tendencia_anos", historico[1]],
        "Base sem média 7 dias": [*anual, "tendencia_anos", historico[0]],
        "Base": base,
        **{f"Base + {c}": [*base, c] for c in candidatas},
        "Base + meteorologia": [*base, *meteo_base],
        "Base + meteorologia e radiação": [*base, *externo],
        "Extensão em vez de precipitação": [
            *base, *[c for c in externo if c != "log_precipitacao"], "areas_com_chuva_pct"
        ],
        "Base + meteorologia, radiação e extensão": [*base, *externo, "areas_com_chuva_pct"],
        "Base + meteorologia, radiação e duração": [*base, *externo, "duracao_dia_h"],
        "Duração em vez de ciclo anual": ["duracao_dia_h", "tendencia_anos", *historico, *externo],
    }


linhas = []
for tipo in TIPOS:
    comparavel = diario.loc[:"2023"].dropna(subset=[tipo, f"{tipo}_d2", f"{tipo}_media7"])
    for ano in anos_validacao:
        ajuste = comparavel[comparavel.index.year < ano]
        validacao = comparavel[comparavel.index.year == ano]
        historica = ajuste.groupby(["mes", "dia_semana"])[tipo].mean()
        chaves = pd.MultiIndex.from_frame(validacao[["mes", "dia_semana"]])
        referencia = historica.reindex(chaves).fillna(ajuste[tipo].mean()).to_numpy(dtype=float)
        previsoes = {"Média histórica": referencia}
        for nome, colunas in variantes(tipo).items():
            previsoes[nome] = ajustar(ajuste, tipo, colunas).predict(matriz(validacao, colunas))
        for nome, previsao in previsoes.items():
            linhas.append({
                "Tipo": tipo, "Ano": ano, "Modelo": nome, "Dias": len(validacao),
                **metricas(validacao[tipo], previsao),
            })

resultados = pd.DataFrame(linhas)
assert resultados["Ano"].max() == 2023
display(resultados.groupby("Ano")["Dias"].first().rename("Dias comparados").to_frame())
mae = resultados.pivot(index=["Tipo", "Ano"], columns="Modelo", values="MAE")
principais = ["Média histórica", "Calendário", "Calendário + histórico", "Base", "Base + meteorologia",
              "Base + meteorologia e radiação", "Base + meteorologia, radiação e duração"]
display(mae[principais].round(3))
display(resultados[resultados["Ano"].eq(2023) & resultados["Modelo"].isin(principais)]
        .pivot(index="Modelo", columns="Tipo", values="Deviância média").round(3))

# Diferenças emparelhadas. Valores positivos indicam redução de MAE face ao comparador.
comparacoes = {
    "Tendência": ("Calendário", "Calendário + tendência"),
    "Tendência dado o histórico": ("Calendário + histórico", "Base"),
    "Histórico sem tendência": ("Calendário", "Calendário + histórico"),
    "Histórico em bloco": ("Calendário + tendência", "Base"),
    "D−2 dado o restante histórico": ("Base sem D−2", "Base"),
    "Média 7 dias dado D−2": ("Base sem média 7 dias", "Base"),
    **{nomes.get(c, "log(1 + precipitação)"): ("Base", f"Base + {c}")
       for c in candidatas},
    "Bloco meteorológico": ("Base", "Base + meteorologia"),
    "Radiação dado o bloco meteorológico": ("Base + meteorologia", "Base + meteorologia e radiação"),
    "Extensão a substituir precipitação": ("Base + meteorologia e radiação", "Extensão em vez de precipitação"),
    "Extensão dada a precipitação": ("Base + meteorologia e radiação",
                                    "Base + meteorologia, radiação e extensão"),
    "Duração dada a meteorologia e radiação": ("Base + meteorologia e radiação",
                                             "Base + meteorologia, radiação e duração"),
    "Duração a substituir ciclo anual": ("Base + meteorologia e radiação", "Duração em vez de ciclo anual"),
}
ganhos = pd.DataFrame({k: mae[a] - mae[b] for k, (a, b) in comparacoes.items()})
ganho_medio = ganhos.groupby(level="Tipo").mean().reindex(TIPOS)
display(ganho_medio.T.round(3))
display(ganhos.xs(2023, level="Ano").reindex(TIPOS).T.round(3))
display(ganhos.gt(0).groupby(level="Tipo").sum().reindex(TIPOS).T
        .rename_axis("Anos com redução de MAE (de 4)"))

### Como tratar a tendência

O tempo decorrido desde 2010 mede tendência de longo prazo. O seno e o cosseno medem o ciclo anual. No GLM, o coeficiente da tendência expressa uma alteração multiplicativa da contagem esperada por ano, condicionada ao calendário e ao histórico. A lacuna de 2020–2022 permanece no calendário, pelo que a distância temporal até 2023 é preservada.

O teste abaixo compara o mesmo modelo com e sem tendência. Se a extrapolação falhar em 2023, o modelo sem tendência é a referência. Uma janela de treino mais recente pode ser testada quando houver um novo bloco de validação. Os anos em falta impedem localizar uma eventual quebra entre 2019 e 2023.

In [ ]:
tendencias = []
for tipo in TIPOS:
    termos = variantes(tipo)["Base"]
    dados = treino.dropna(subset=termos)
    ajustado = ajustar(dados, tipo, termos)
    ganho = ganhos["Tendência dado o histórico"].loc[tipo]
    tendencias.append({
        "Tipo": tipo,
        "Variação ajustada por ano (%)": 100 * np.expm1(ajustado.params["tendencia_anos"]),
        "Redução de MAE média (4 validações)": ganho.mean(),
        "Redução de MAE em 2023": ganho.loc[2023],
        "Anos com redução": ganho.gt(0).sum(),
    })
display(pd.DataFrame(tendencias).set_index("Tipo").round(3))

## 5. Diagnósticos do modelo de contagens

As respostas têm de ser inteiras e não negativas, como verificado na secção 1. O GLM de Poisson assume que a média condicional é positiva, com ligação log, e que a variância condicional é igual à média. A normalidade da resposta e a variância constante ao longo dos valores previstos pertencem aos pressupostos de outros modelos. Para Poisson verificamos convergência, posto da matriz, colinearidade, dispersão condicional, zeros e ajuste da média. A autocorrelação residual é investigada para aferir a independência condicional.

Os diagnósticos abaixo ajustam o bloco compacto em 2010–2019 e verificam a inclusão de duração do dia em separado. O VIF mede a inflação da variância dos coeficientes por redundância linear. Um grande VIF dificulta interpretar efeitos individuais. O rácio de Pearson acima de 1 sugere sobredispersão. A comparação de zeros usa a média de exp(−μ), a probabilidade de zero do Poisson ajustado em cada dia.

In [ ]:
diagnosticos, vifs = [], []
fig, eixos = plt.subplots(3, 2, figsize=(11.5, 9), layout="constrained")
for i, tipo in enumerate(TIPOS):
    colunas = variantes(tipo)["Base + meteorologia e radiação"]
    dados = treino.dropna(subset=colunas)
    modelo = ajustar(dados, tipo, colunas)
    residuos = pd.Series(modelo.resid_pearson, index=dados.index)
    assert residuos.index.to_series().diff().dropna().eq(pd.Timedelta(days=1)).all()
    mu = modelo.fittedvalues
    lb = acorr_ljungbox(residuos, lags=[7, 14, 28], return_df=True)
    influencia = modelo.get_influence().cooks_distance[0]
    diagnosticos.append({
        "Tipo": tipo, "n": len(dados), "Convergiu": modelo.converged,
        "Pearson / gl": modelo.pearson_chi2 / modelo.df_resid,
        "Zeros observados (%)": 100 * dados[tipo].eq(0).mean(),
        "Zeros Poisson (%)": 100 * np.exp(-mu).mean(),
        "ACF D−1": residuos.autocorr(1), "ACF D−7": residuos.autocorr(7),
        "Ljung–Box p (28)": lb.loc[28, "lb_pvalue"],
        "Cook máximo": np.max(influencia),
    })
    for nome in ["Base + meteorologia e radiação", "Base + meteorologia, radiação e duração"]:
        x = matriz(dados, variantes(tipo)[nome])
        for j, coluna in enumerate(x.columns):
            if coluna != "const":
                vifs.append({"Tipo": tipo, "Modelo": nome, "Entrada": coluna,
                             "VIF": variance_inflation_factor(x.to_numpy(), j)})

    lags = np.arange(1, 29)
    eixos[i, 0].stem(lags, [residuos.autocorr(int(k)) for k in lags], basefmt=" ")
    eixos[i, 0].axhline(0, color="#666", lw=.8)
    eixos[i, 0].set(title=f"{tipo} · autocorrelação residual", xlabel="Atraso em dias", ylabel="ACF")

    calibracao = pd.DataFrame({"Previsto": mu, "Observado": dados[tipo]})
    resumo_cal = calibracao.groupby(pd.qcut(mu, 10, duplicates="drop"), observed=True).mean()
    eixos[i, 1].scatter(resumo_cal["Previsto"], resumo_cal["Observado"], color=CORES[tipo])
    minimo, maximo = resumo_cal.min().min(), resumo_cal.max().max()
    eixos[i, 1].plot([minimo, maximo], [minimo, maximo], ls="--", color="#777")
    eixos[i, 1].set(title=f"{tipo} · ajuste médio por decis de previsão",
                    xlabel="Contagem média ajustada", ylabel="Contagem média observada")

tabela_diagnosticos = pd.DataFrame(diagnosticos).set_index("Tipo")
display(tabela_diagnosticos.style.format({
    "Pearson / gl": "{:.3f}", "Zeros observados (%)": "{:.2f}", "Zeros Poisson (%)": "{:.2f}",
    "ACF D−1": "{:.3f}", "ACF D−7": "{:.3f}", "Ljung–Box p (28)": "{:.2e}", "Cook máximo": "{:.3f}",
}))
vif = pd.DataFrame(vifs)
display(vif.groupby(["Modelo", "Tipo"])["VIF"].max().unstack().round(2))
display(vif[vif["VIF"].ge(5)].sort_values("VIF", ascending=False).round(2))
plt.show()

### Forma das relações na escala log

Para perceber se um termo linear em cada preditor contínuo é suficiente como primeira aproximação, testamos em 2023 a adição de um termo quadrático por vez ao modelo compacto. O centro e a escala são obtidos em 2010–2019. A avaliação usa o mesmo conjunto de dias por tipo. Este diagnóstico ajuda a motivar modelos flexíveis se estes melhorarem o erro fora do treino.

In [ ]:
curvatura = []
for tipo in TIPOS:
    colunas = variantes(tipo)["Base + meteorologia e radiação"]
    ajuste = treino.dropna(subset=colunas)
    val = diario.loc["2023"].dropna(subset=colunas)
    mae_linear = metricas(val[tipo], ajustar(ajuste, tipo, colunas).predict(matriz(val, colunas)))["MAE"]
    for candidata in [*meteo_base, "radiacao_total_mj_m2", f"{tipo}_media7"]:
        centro, escala = ajuste[candidata].mean(), ajuste[candidata].std()
        ajuste_q = ajuste.assign(curvatura=((ajuste[candidata] - centro) / escala) ** 2)
        val_q = val.assign(curvatura=((val[candidata] - centro) / escala) ** 2)
        termos = [*colunas, "curvatura"]
        mu = ajustar(ajuste_q, tipo, termos).predict(matriz(val_q, termos))
        curvatura.append({"Tipo": tipo, "Entrada": candidata,
                          "Redução de MAE em 2023": mae_linear - metricas(val_q[tipo], mu)["MAE"]})
display(pd.DataFrame(curvatura).round(3))

## 6. O que entra na próxima fase

A referência principal é a média histórica por mês e dia da semana, estimada apenas a partir do treino. A regressão de Poisson é a referência modelada para comparar com a rede neuronal. Os valores abaixo são resultados retrospetivos em validação.

In [ ]:
linhas = []
for tipo in TIPOS:
    dados = mae.loc[(tipo, 2023)]
    linhas.append(
        f"- **{tipo}**. MAE em 2023: {dados['Média histórica']:.2f} na média histórica, "
        f"{dados['Base']:.2f} em Poisson com calendário, tendência e histórico, "
        f"{dados['Base + meteorologia e radiação']:.2f} após acrescentar meteorologia e radiação. "
        f"Rácio de dispersão condicional no treino: {tabela_diagnosticos.loc[tipo, 'Pearson / gl']:.2f}."
    )
display(Markdown("\n".join(linhas)))

### Entradas a levar para a modelação

- **Calendário e histórico.** Usar dia da semana, ciclo anual e média recente para os três tipos. Manter D−2 como entrada candidata, com evidência mais consistente nos despistes. Testar a tendência para colisões e despistes, onde ajudou em três dos quatro anos dado o histórico. Nos atropelamentos, a tendência trouxe um ganho médio praticamente nulo.
- **Meteorologia.** Para colisões, testar precipitação e uma medida de luz ou cobertura de nuvens de cada vez. Nebulosidade e radiação melhoraram o MAE quando avaliadas isoladamente em cada um dos quatro anos. Para despistes, a temperatura teve um ganho pequeno nos quatro anos, enquanto o bloco meteorológico completo piorou em 2023. Para atropelamentos, começar pela base de calendário e histórico e exigir um ganho consistente para alargar o modelo. Os resultados para vento ficaram próximos de zero ou foram desfavoráveis.
- **Redundância.** Tratar precipitação total e extensão da chuva como alternativas iniciais (Spearman de 0,94). Fazer o mesmo com horas de sol direto e radiação (0,97). Usar o ciclo anual como representação principal da sazonalidade. Acrescentar duração do dia a esse ciclo elevou o VIF para mais de 1.000, com pouco ganho consistente na validação.
- **Pressupostos.** O Poisson compacto apresentou sobredispersão nos três tipos e autocorrelação residual mais acentuada nos despistes. Os valores são úteis para comparação de previsões pontuais. Antes de usar o modelo para intervalos ou inferência, avaliar uma binomial negativa e métodos que considerem a dependência temporal.

Para o teste final, congelar as definições de entradas e de modelos antes de avaliar 2024. Medir MAE e deviância por tipo nas mesmas datas e comparar a rede e a regressão com as mesmas entradas. Para responder à pergunta sobre as condições do dia em contexto operacional, repetir a comparação com previsões meteorológicas arquivadas que estivessem disponíveis antes do dia-alvo.

**Referências metodológicas.** [Poisson no statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.genmod.families.family.Poisson.html) · [Validação temporal com variáveis atrasadas](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).